# 3 5 count analysis moral outrage

Peer-review copy. Inputs are pseudonymized processed data; see README.md and DATA_DICTIONARY.md. Original preparation/collection steps are outside this package.

In [ ]:
from review_setup import prepare_workspace
prepare_workspace()
from review_setup import initialize_stata
initialize_stata()


In [ ]:
import pandas as pd
from utils import get_source_corrections, StataDiD

source_corrections = get_source_corrections(window_days=28).query('current_correction==1').reset_index(drop=True)

In [ ]:
# Reply-level outrage scores with a precomputed Before/After indicator.
# Retain individual scores for the original event-clustered comparison.
fact_replies = pd.read_parquet('data/fact_replies.parquet')
fact_replies = fact_replies.merge(source_corrections[['tweet_id']], on='tweet_id', how='inner')
print(fact_replies.shape)
fact_replies.head()


In [ ]:
from utils import mean_ci

results = mean_ci(
    fact_replies,
    score='moral_outrage_probability',
    group='displayed',
    cluster='tweet_id'
)

display(results['summary'].round(3))
display(results['differences'].round(3))
display(results['overall'].to_frame('Value').round(3))

In [ ]:
outrage_change = fact_replies.groupby(['tweet_id', 'displayed'])['moral_outrage_probability'].mean().unstack().reset_index().dropna()
outrage_change['outrage_change'] = outrage_change['After'] - outrage_change['Before']
outrage_change['outrage_change'].describe()

In [ ]:
source_corrections = source_corrections.merge(outrage_change[['tweet_id', 'outrage_change']], on='tweet_id', how='inner')
print(source_corrections.shape)
source_corrections.head()

In [ ]:
source_corrections['treated'] = (source_corrections['outrage_change']>0).astype(int)

### Original post daily [-4,4]

In [ ]:
ts_posts = pd.read_parquet('data/ts_posts_daily.parquet.gzip').query('-4<=days_from_display<=4').reset_index(drop=True)
ts_posts = ts_posts.merge(source_corrections, on='tweet_id', how='inner')
print(ts_posts.shape)
ts_posts.head()

In [ ]:
stata_did = StataDiD(ts_posts, 'treated', 'tweet_id', 'days_from_display', -4, 4)

In [ ]:
stata_did.multi_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
from utils import get_coefs
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
from utils import att_plot
ylims = (-0.262, 0.462)
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att, ll_agg_att, hl_agg_att, 'Daily', 'Days from note display', 'Estimated DiD estimate\n(higher vs. lower outrage)', 'did_posts_daily_outrage', ylims=ylims, legend_ncol=1, legend_title='First correction')

### Original post weekly [-4,4]

In [ ]:
ts_posts = pd.read_parquet('data/ts_posts_weekly.parquet.gzip')
ts_posts = ts_posts.merge(source_corrections, on='tweet_id', how='inner')
print(ts_posts.shape)
ts_posts.head()

In [ ]:
stata_did = StataDiD(ts_posts, 'treated', 'tweet_id', 'weeks_from_display', -4, 4)

In [ ]:
stata_did.multi_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
from utils import get_coefs
means_mp_atts, lls_mp_atts, hls_mp_atts = get_coefs(reg='ppmlhdfe', start_line=8, end_line=15, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
stata_did.tw_did(reg='ppmlhdfe', dev='tweet_count', cluster='account_cluster')

In [ ]:
mean_agg_att, ll_agg_att, hl_agg_att = get_coefs(reg='ppmlhdfe', start_line=2, end_line=3, stat_type='z', irr_or=1, unit_group=True)

In [ ]:
from utils import att_plot
att_plot(-4, 4, 4, means_mp_atts, lls_mp_atts, hls_mp_atts, mean_agg_att, ll_agg_att, hl_agg_att, 'Weekly', 'Weeks from note display', 'Estimated DiD effect\n(higher vs. lower outrage)', 'did_posts_weekly_outrage', ylims=ylims, legend_ncol=1, legend_title='First correction')